In [ ]:
import os, gc, ctypes
import pyarrow as pa
import pyarrow.parquet as pq
import xgboost as xgb
import polars as pl
import numpy as np
from tqdm.auto import tqdm

In [ ]:
try:
    libc = ctypes.CDLL("libc.so.6")
    def trim_memory():
        libc.malloc_trim(0)
except:
    def trim_memory():
        pass

print("Đang tạo bảng đặc trưng nền bằng Polars...")
lf_train = pl.scan_parquet(TRAIN_PATH)
lf_meta = pl.scan_parquet(META_PATH)

df_use = (
    lf_train.group_by('mapped_user_id').len()
    .rename({"len": "user_orders"})
    .with_columns(pl.col("user_orders").cast(pl.Float32))
    .collect()
)

df_ite = (
    lf_train.group_by('mapped_item_id').len()
    .rename({"len": "item_sales"})
    .with_columns(pl.col("item_sales").cast(pl.Float32))
    .collect()
)

mapping_df = lf_train.select(['parent_asin', 'mapped_item_id']).unique()
df_price = (
    lf_meta.select(['parent_asin', 'price'])
    .join(mapping_df, on='parent_asin', how='inner')
    .with_columns(
        pl.col('price').str.replace_all(r'[^0-9.]', '').cast(pl.Float32, strict=False).fill_null(0.0)
    )
    .select(['mapped_item_id', 'price'])
    .unique()
    .collect()
)

del lf_train, lf_meta, mapping_df
gc.collect()
trim_memory()

print("Bắt đầu chấm điểm (Chiến thuật Continuous Streaming Flush)...")
model.set_param({"device": "cuda"})

pf = pq.ParquetFile(CAND_PATH)
print(f"Tổng số dòng sẽ xử lý: {pf.metadata.num_rows:,}")

reader = pf.iter_batches(batch_size=500000)
buffer_df = pl.DataFrame()
writer = None
FINAL_TOP100_PATH = '/kaggle/working/top100_final_recommendations.parquet'

for batch in tqdm(reader, desc="Scoring Chunks"):
    chunk = pl.from_arrow(batch)
    
    # Nối đặc trưng
    chunk = chunk.join(df_use, on='mapped_user_id', how='left')
    chunk = chunk.join(df_ite, on='mapped_item_id', how='left')
    chunk = chunk.join(df_price, on='mapped_item_id', how='left')
    
    chunk = chunk.with_columns([
        pl.col('user_orders').fill_null(0.0),
        pl.col('item_sales').fill_null(0.0),
        pl.col('price').fill_null(0.0)
    ])
    
    # Chấm điểm an toàn với DMatrix
    X_cands = chunk.select(FEATURES).to_numpy()
    dtest = xgb.DMatrix(X_cands, missing=np.nan, feature_names=FEATURES)
    scores = model.predict(dtest)
    
    # Tạo bảng cực nhẹ
    chunk_res = pl.DataFrame({
        'mapped_user_id': chunk['mapped_user_id'],
        'mapped_item_id': chunk['mapped_item_id'],
        'score': pl.Series(scores, dtype=pl.Float32)
    })
    
    # Nạp vào ống đệm
    if buffer_df.height > 0:
        buffer_df = pl.concat([buffer_df, chunk_res])
    else:
        buffer_df = chunk_res
        
    del chunk, X_cands, dtest, scores, chunk_res
    gc.collect()
    
    # Lấy User cuối cùng (có thể đang bị cắt dở)
    last_user = buffer_df.get_column('mapped_user_id')[-1]
    
    # Phân tách: Những User đã hoàn tất và User cuối cùng
    completed_users_df = buffer_df.filter(pl.col('mapped_user_id') != last_user)
    buffer_df = buffer_df.filter(pl.col('mapped_user_id') == last_user)
    
    # Nếu có User hoàn tất, lập tức cắt Top 100 và ghi thẳng xuống đĩa cứng
    if completed_users_df.height > 0:
        top100_completed = (
            completed_users_df
            .sort(['mapped_user_id', 'score'], descending=[False, True])
            .group_by('mapped_user_id')
            .head(100)
        )
        
        table = top100_completed.to_arrow()
        if writer is None:
            writer = pq.ParquetWriter(FINAL_TOP100_PATH, table.schema)
        writer.write_table(table)
        
        del completed_users_df, top100_completed, table
        gc.collect()
        
    trim_memory()

# Xử lý phần cặn còn lại của ống đệm sau khi hết vòng lặp
if buffer_df.height > 0:
    top100_last = (
        buffer_df
        .sort(['mapped_user_id', 'score'], descending=[False, True])
        .group_by('mapped_user_id')
        .head(100)
    )
    table = top100_last.to_arrow()
    if writer is None:
        writer = pq.ParquetWriter(FINAL_TOP100_PATH, table.schema)
    writer.write_table(table)

if writer is not None:
    writer.close()

print(f"ĐÃ LƯU KẾT QUẢ TẠI: {FINAL_TOP100_PATH}")